# Jornada completa — Planilha Turmas (DEMO)

Execute **Restart Kernel and Run All**. Dados sintéticos reproduzíveis (seed 42), sem fonte institucional. Métricas demonstram integração, não qualidade real. O grão é curso/período/disciplina/ciclo. A demanda histórica do período é atribuída às disciplinas pela regra provisória da POC; não é decisão funcional aprovada do MER.

O notebook abre SQL com usuário de leitura; usa MLflow remoto em proxy, recarrega uma versão explícita e grava uma proposta DRAFT em transação. Não promove modelos. Cada Run All cria uma nova execução; repetir apenas a persistência com o mesmo conteúdo é idempotente.


## 1. Configurações e dependências

Testar SQL de origem/destino, versão do SDK/servidor, tracking real e upload/download pelo proxy antes de qualquer fit.

**Esperado/evidência:** Tabela PASS; se houver FAIL o notebook interrompe com diagnóstico, sem tracking local.


In [1]:
import os, uuid, time, json
from pathlib import Path
import pandas as pd
import numpy as np
import httpx
import mlflow
import mlflow.sklearn
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
from src.workbench.journey_steps import dependencies, require_dependencies, step
from src.common.tracing import configure_tracing, trace_headers, trace_evidence
from src.workbench.verification import verify_model_storage, verify_proposal_sql
cfg = Settings.load()
execution_id = str(uuid.uuid4())
statuses = []
checks = dependencies(cfg, execution_id)
display(pd.DataFrame(checks))
require_dependencies(checks, cfg)
experiment_id = configure_tracing(cfg)
display({"experiment":cfg.experiment,"experiment_id":experiment_id,"model_name":cfg.model_name})
statuses.append(dict(stage="dependencies", status="PASS"))
display({"execution_id":execution_id, "mode":cfg.mode, "tracking":cfg.tracking})

,dependency,status,attempts
0,source SQL,PASS,1
1,application SQL,PASS,1
2,MLflow PostgreSQL + proxy artifact roundtrip,PASS,1


{'experiment': 'planilha-turmas-proxy-v1',
 'experiment_id': '4',
 'model_name': 'planilha-turmas-demo'}

{'execution_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'mode': 'DEMO',
 'tracking': 'http://mlflow:5000'}

## 2. Extração SQL versionada

A conexão é aberta por src.data.journey usando SOURCE_DATABASE_URL. Fonte DEMO: planejamento_db.planning.historical_enrollments e catálogo relacionado. O seed já foi carregado no bootstrap.

**Esperado/evidência:** Consulta parametrizada, snapshot/hash, filtros, contagens, tipos e amostra sem credenciais.


In [2]:
from src.data.journey import extract, prepare, FEATURES, TARGET, QUERY
with step("SQL extraction", execution_id, statuses):
    history, snapshot = extract(cfg)
    display(Markdown("```sql\n" + QUERY.read_text() + "\n```"))
    display({"source":"planejamento_db/planning/SYNTHETIC", "cycle":cfg.cycle,
        "course":cfg.course or "all", "snapshot":snapshot, "rows":len(history),
        "nulls":int(history.isna().sum().sum()), "duplicates":int(history.duplicated().sum())})
    display(history.dtypes.astype(str).rename("type").to_frame())
    display(history.head())

{"timestamp": "2026-11-04T19:24:21.106296+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "SQL extraction", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


```sql
-- journey-history-v1. Target: total students per period, replicated to its disciplines
-- under the provisional POC rule: every discipline receives the whole cohort.
SELECT h.course_id::text, h.period_id::text, pd.discipline_id::text,
       h.reference_year * 2 + h.reference_semester - 1 AS semester_index,
       h.reference_semester AS semester,
       h.regular_students + h.dependency_students + h.elective_students AS students
FROM planning.historical_enrollments h
JOIN planning.dataset_snapshots s ON s.id = h.snapshot_id
JOIN planning.academic_cycles c ON c.id = s.cycle_id
JOIN planning.period_disciplines pd ON pd.period_id = h.period_id
WHERE h.snapshot_id = CAST(:snapshot AS uuid) AND c.cycle_code = :cycle
  AND s.sealed_at IS NOT NULL
  AND (CAST(:course AS text) IS NULL OR h.course_id::text = :course)
  AND h.reference_year * 2 + h.reference_semester - 1 < c.target_year * 2 + c.target_semester - 1
ORDER BY semester_index, h.course_id, h.period_id, pd.discipline_id

```

{'source': 'planejamento_db/planning/SYNTHETIC',
 'cycle': '2026-1',
 'course': 'all',
 'snapshot': {'id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
  'cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
  'target_index': 4052,
  'target_semester': 1,
  'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
  'query_sha256': 'defc10d1b6ea752600af271285c54107663e91db9df13914aea7e42613c83adf',
  'query_version': 'journey-history-v1'},
 'rows': 1050,
 'nulls': 0,
 'duplicates': 0}

,type
course_id,object
period_id,object
discipline_id,object
semester_index,int64
semester,int64
students,int64


,course_id,period_id,discipline_id,semester_index,semester,students
0,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,4040,1,86
1,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,4040,1,86
2,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,4040,1,86
3,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,4040,1,82
4,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,4040,1,82


{"timestamp": "2026-11-04T19:24:21.305565+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "SQL extraction", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 0.199, "attempt": 1, "trace_id": "tr-4222edf555191c6c97a244cb6330e788"}


Trace(trace_id=tr-4222edf555191c6c97a244cb6330e788)

## 3. Preparação temporal

Reservar o último semestre histórico para avaliação. O conjunto de proposta contém somente chaves acadêmicas e o próximo semestre, sem alvo. Transformações serão ajustadas exclusivamente no treino.

**Esperado/evidência:** Treino < avaliação < proposta no tempo.


In [3]:
with step("prepare", execution_id, statuses):
    train, evaluation, future = prepare(history, snapshot)
    display(pd.DataFrame([
        {"dataset":"train", "rows":len(train), "start":int(train.semester_index.min()), "end":int(train.semester_index.max())},
        {"dataset":"evaluation", "rows":len(evaluation), "start":int(evaluation.semester_index.min()), "end":int(evaluation.semester_index.max())},
        {"dataset":"proposal", "rows":len(future), "start":int(future.semester_index.min()), "end":int(future.semester_index.max())}]))

{"timestamp": "2026-11-04T19:24:21.312717+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "prepare", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


,dataset,rows,start,end
0,train,960,4040,4050
1,evaluation,90,4051,4051
2,proposal,90,4052,4052


{"timestamp": "2026-11-04T19:24:21.358046+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "prepare", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 0.045, "attempt": 1, "trace_id": "tr-e3bbd67a52793a1edf81fca8c6fa2614"}


Trace(trace_id=tr-e3bbd67a52793a1edf81fca8c6fa2614)

## 4. Modelo e métricas

Ridge(alpha=1, seed=42), one-hot de curso/período/disciplina e padronização temporal. Alvo: estudantes do período atribuídos à disciplina. MAE mede erro absoluto médio; RMSE penaliza mais os erros grandes. Ambas em estudantes.

**Esperado/evidência:** Pipeline instanciado e funções de avaliação definidas antes do fit.


In [4]:
from src.training.journey import estimator, evaluate, SEED
with step("model and metrics", execution_id, statuses):
    model = estimator()
    display({"features":FEATURES, "target":TARGET, "seed":SEED, "metrics":["MAE", "RMSE"]})
    display(model)

{"timestamp": "2026-11-04T19:24:21.704595+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "model and metrics", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


{'features': ['course_id',
  'period_id',
  'discipline_id',
  'semester_index',
  'semester'],
 'target': 'students',
 'seed': 42,
 'metrics': ['MAE', 'RMSE']}

Pipeline(steps=[('features',
                 ColumnTransformer(transformers=[('academic',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['course_id', 'period_id',
                                                   'discipline_id']),
                                                 ('time', StandardScaler(),
                                                  ['semester_index',
                                                   'semester'])])),
                ('regression', Ridge(random_state=42))])

{"timestamp": "2026-11-04T19:24:21.748770+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "model and metrics", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 0.044, "attempt": 1, "trace_id": "tr-0ea4a6b4b33d6cb4ef6e01df7e3db1f6"}


Trace(trace_id=tr-0ea4a6b4b33d6cb4ef6e01df7e3db1f6)

## 5. Experimento e execução

Criar/recuperar o experimento proxy e registrar identidade da extração, query, código, features e referência sintética. Cada etapa posterior encerra a run como FAILED se ocorrer erro.

**Esperado/evidência:** experiment_id/run_id e link clicável; CSV de referência e SQL enviados pelo proxy.


In [5]:
from src.mlops.journey import start, register
with step("tracking", execution_id, statuses):
    trace = start(cfg, execution_id, snapshot, history)
    display(trace)
    display(Markdown(f"[Abrir execução no MLflow]({trace['run_url']})"))

{"timestamp": "2026-11-04T19:24:21.759098+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "tracking", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


{'run_id': 'c0d9f3138c874049b339e0f54ab8348c',
 'experiment_id': '4',
 'run_url': 'http://127.0.0.1:5000/#/experiments/4/runs/c0d9f3138c874049b339e0f54ab8348c'}

[Abrir execução no MLflow](http://127.0.0.1:5000/#/experiments/4/runs/c0d9f3138c874049b339e0f54ab8348c)

{"timestamp": "2026-11-04T19:24:21.910775+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "tracking", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 0.152, "attempt": 1, "trace_id": "tr-42410e8fafa17c97720af61ba16bc7c7"}


Trace(trace_id=tr-42410e8fafa17c97720af61ba16bc7c7)

## 6. Treinamento e avaliação

Executar fit/predict reais. O modelo persistido é exatamente o avaliado; não há retreino oculto com o conjunto de avaliação.

**Esperado/evidência:** MAE/RMSE calculados e registrados; métricas DEMO não comprovam qualidade institucional.


In [6]:
if mlflow.active_run() is None or mlflow.active_run().info.run_id != trace["run_id"]:
    raise RuntimeError("Treinamento exige a run desta jornada ativa. Para novo treino, execute desde a primeira etapa.")
with step("fit and evaluate", execution_id, statuses):
    started = time.monotonic()
    model.fit(train[FEATURES], train[TARGET])
    predicted = model.predict(evaluation[FEATURES])
    metrics = evaluate(evaluation[TARGET], predicted)
    mlflow.log_metrics({**metrics, "training_seconds":time.monotonic()-started})
    mlflow.log_dict({"actual":evaluation[TARGET].tolist(), "predicted":predicted.tolist()}, "evaluation.json")
    display(metrics)

{"timestamp": "2026-11-04T19:24:21.944355+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "fit and evaluate", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


{'mae': 3.758630658273197, 'rmse': 4.9142272796917865}

{"timestamp": "2026-11-04T19:24:22.041627+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "fit and evaluate", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 0.097, "attempt": 1, "trace_id": "tr-6729e65a3bdc382eccf04fa0dd3d74ce"}


Trace(trace_id=tr-6729e65a3bdc382eccf04fa0dd3d74ce)

## 7. Persistência do modelo e Registry

O SDK envia modelo, transformações, assinatura e exemplo ao MLflow; somente o servidor grava o RustFS. A versão será selecionada explicitamente para teste DRAFT, sem alias champion.

**Esperado/evidência:** Nome, versão, URI e run; objeto treinado removido da memória.


In [7]:
with step("register", execution_id, statuses):
    trace = register(cfg, globals().get("model"), train[FEATURES].head(3), trace, execution_id)
    mlflow.end_run(status="FINISHED")
    display(trace)
    globals().pop("model", None)  # Infer?ncia recarrega a vers?o persistida.

{"timestamp": "2026-11-04T19:24:22.047903+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "register", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


/opt/conda/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/04 19:24:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during de

Registered model 'planilha-turmas-demo' already exists. Creating a new version of this model...
2026/11/04 19:24:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: planilha-turmas-demo, version 15


Created version '15' of model 'planilha-turmas-demo'.


{'run_id': 'c0d9f3138c874049b339e0f54ab8348c',
 'experiment_id': '4',
 'run_url': 'http://127.0.0.1:5000/#/experiments/4/runs/c0d9f3138c874049b339e0f54ab8348c',
 'model_name': 'planilha-turmas-demo',
 'model_version': '15',
 'model_uri': 'models:/planilha-turmas-demo/15',
 'artifact_uri': 'models:/m-342590e0c9ca494e99e6d746160225f8'}

{"timestamp": "2026-11-04T19:24:23.207284+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "register", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 1.159, "attempt": 1, "trace_id": "tr-1c61032c4f969e14529ed85397d18177"}


Trace(trace_id=tr-1c61032c4f969e14529ed85397d18177)

## 7b. Conferir o modelo no Object Store

Leitura independente no RustFS e pelo proxy MLflow, sem gravar uma segunda c?pia. **Esperado:** bytes e SHA256 id?nticos para modelo, assinatura e depend?ncias; bucket/chave exibidos. Pode repetir esta verifica??o.


**Esperado/evidência:** PASS; evid?ncia exibida.


In [8]:
with step("verify Object Store", execution_id, statuses):
    storage_evidence = verify_model_storage(trace)
    display(pd.DataFrame(storage_evidence))

{"timestamp": "2026-11-04T19:24:23.213501+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "verify Object Store", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


,bucket,key,bytes,sha256,object_version_id,model_version,status
0,pucpr-poc,mlflow/4/models/m-342590e0c9ca494e99e6d7461602...,1336,752dfba474fc343bf7f6654ed174c08207af206086532c...,9d81a473-df96-4333-9fe9-1e6ee9a5d1e1,15,PASS
1,pucpr-poc,mlflow/4/models/m-342590e0c9ca494e99e6d7461602...,8671,4268666504b258b661b9c6cc009bdf8c21d06fde4317fb...,f0e9a40e-7cc0-4ad3-af4d-a358027084a8,15,PASS
2,pucpr-poc,mlflow/4/models/m-342590e0c9ca494e99e6d7461602...,80,3f22b46a5a2293ca36743e1757d14e9968aa14f9d57a28...,d857cedd-8e60-4513-b5ab-8ee094633e90,15,PASS


{"timestamp": "2026-11-04T19:24:23.504306+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "verify Object Store", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 0.291, "attempt": 1, "trace_id": "tr-1885e64cc3ee4d33e08f9eb6e6c11d90"}


Trace(trace_id=tr-1885e64cc3ee4d33e08f9eb6e6c11d90)

## 8. Recarregar e inferir no serviço existente

Baixar a versão registrada e executar predict no conjunto futuro. Chamar também o serviço inference: ele carrega a mesma versão e verifica run/execução/snapshot. Registro não significa promoção operacional.

**Esperado/evidência:** Download real do artefato; serviço carregou versão igual e produziu previsões iguais.


In [9]:
from src.inference.journey import load_version
from src.common.settings import required
with step("reload and inference", execution_id, statuses):
    restored, registered = load_version(trace["model_name"], trace["model_version"], trace["run_id"], execution_id)
    predictions = restored.predict(future[FEATURES])
    response = httpx.post(cfg.inference + "/internal/draft/predict", timeout=120,
        headers={"Authorization":"Bearer " + required("INFERENCE_TOKEN"), "X-Correlation-ID":execution_id, **trace_headers()},
        json={"execution_id":execution_id, "snapshot_id":snapshot["id"], "model_name":trace["model_name"],
              "model_version":trace["model_version"], "run_id":trace["run_id"], "records":future[FEATURES].to_dict("records")})
    response.raise_for_status()
    inference = response.json()
    trace["inference_id"] = inference["inference_id"]
    trace["artifact_sha256"] = next(item["sha256"] for item in storage_evidence if item["key"].endswith("/model.pkl"))
    assert inference["model_version"] == trace["model_version"] and inference["run_id"] == trace["run_id"]
    np.testing.assert_allclose(predictions, inference["predictions"], rtol=1e-10)
    client = mlflow.MlflowClient()
    client.log_metric(trace["run_id"], "inference_rows", len(predictions))
    client.log_metric(trace["run_id"], "inference_seconds", inference["duration_seconds"])
    display({k:v for k,v in inference.items() if k != "predictions"})
    display(future.assign(forecast_raw=predictions).head())

{"timestamp": "2026-11-04T19:24:23.518708+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "reload and inference", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


{'inference_id': 'be5cf3b8-2839-4658-bc50-e8d24d4c2add',
 'execution_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'run_id': 'c0d9f3138c874049b339e0f54ab8348c',
 'model_name': 'planilha-turmas-demo',
 'model_version': '15',
 'loaded': True,
 'operationally_promoted': False,
 'duration_seconds': 0.5829395379987545}

,course_id,period_id,discipline_id,semester_index,semester,forecast_raw
0,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,4052,1,102.089955
1,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,4052,1,102.089955
2,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,4052,1,102.089955
3,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,4052,1,103.245511
4,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,4052,1,103.245511


{"timestamp": "2026-11-04T19:24:24.558076+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "reload and inference", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 1.039, "attempt": 1, "trace_id": "tr-f94aea0befa7ae596ba1c43bd3fc8d94"}


Trace(trace_id=tr-f94aea0befa7ae596ba1c43bd3fc8d94)

## 9. Compor Planilha Turmas

Regra provisória POC: estudantes propostos = max(0, ceil(previsão)); turmas = ceil(estudantes/capacidade). Capacidade demonstrativa = 40. Zero demanda gera zero turmas. Sem ajustes humanos nesta execução.

**Esperado/evidência:** Valor previsto, estudantes propostos, turmas e ajuste humano separados.


In [10]:
from src.persistence.journey import compose, persist, RULE
with step("compose proposal", execution_id, statuses):
    capacity = 40
    lines = compose(future, predictions, capacity)
    display({"rule":RULE, "capacity":capacity, "state":"DRAFT"})
    display(pd.DataFrame(lines).head())

{"timestamp": "2026-11-04T19:24:24.564961+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "compose proposal", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


{'rule': 'POC-whole-cohort-ceil-capacity-v1', 'capacity': 40, 'state': 'DRAFT'}

,course_id,period_id,discipline_id,forecast_raw,proposed_students,proposed_groups,human_adjustment
0,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,102.089955,103,3,None
1,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,102.089955,103,3,None
2,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,102.089955,103,3,None
3,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,103.245511,104,3,None
4,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,103.245511,104,3,None


{"timestamp": "2026-11-04T19:24:24.609196+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "compose proposal", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 0.044, "attempt": 1, "trace_id": "tr-aefa5756077a436d82fecfd549b42358"}


Trace(trace_id=tr-aefa5756077a436d82fecfd549b42358)

## 10. Persistir e verificar idempotência

Gravar cabeçalho e todas as linhas numa transação PostgreSQL em planejamento_db.planning.proposal_*. Repetição idêntica retorna a geração existente; conteúdo diferente com o mesmo UUID é rejeitado. Novo Run All preserva as versões anteriores.

**Esperado/evidência:** Geração DRAFT persistida com vínculos SQL/run/modelo/regra; repetição sem duplicações.


In [11]:
with step("persist and idempotency", execution_id, statuses):
    generation = persist(cfg, execution_id, snapshot, trace, lines, capacity)
    repeated = persist(cfg, execution_id, snapshot, trace, lines, capacity)
    assert generation["execution_id"] == repeated["execution_id"]
    assert generation["row_count"] == len(lines)
    from src.persistence.lineage import publish_manifest
    lineage_manifest = publish_manifest(cfg, execution_id)
    display(lineage_manifest)
    display(generation)

{"timestamp": "2026-11-04T19:24:24.616046+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "persist and idempotency", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


{'uri': 's3://pucpr-poc/lineage/poc/planilha-turmas/proposals/712b11cd-a96a-4bb9-83bb-691724ceae88/revisions/1/manifest.json',
 'sha256': '5d95db9917fea98b06c5d151f370de81c6c943039eaaccb9ce86d8524cd690fb'}

{'execution_id': UUID('712b11cd-a96a-4bb9-83bb-691724ceae88'),
 'correlation_id': UUID('712b11cd-a96a-4bb9-83bb-691724ceae88'),
 'cycle_id': UUID('0fa6938d-2743-56ae-a057-dffd2274d90a'),
 'snapshot_id': UUID('fb931e1e-43d0-5ff8-938c-0d2ba4a68794'),
 'course_scope': None,
 'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
 'payload_sha256': '406a0f2c6047a48db409ae4a58f3c3cbe2a84294adbfc03a8e5bb5261234bed9',
 'run_id': 'c0d9f3138c874049b339e0f54ab8348c',
 'experiment_id': '4',
 'model_name': 'planilha-turmas-demo',
 'model_version': '15',
 'artifact_uri': 'models:/m-342590e0c9ca494e99e6d746160225f8',
 'rule_version': 'POC-whole-cohort-ceil-capacity-v1',
 'capacity': 40,
 'mode': 'DEMO',
 'status': 'DRAFT',
 'row_count': 90,
 'generated_at': datetime.datetime(2026, 11, 4, 19, 24, 24, 629371, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC'))}

{"timestamp": "2026-11-04T19:24:24.738354+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "persist and idempotency", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 0.122, "attempt": 1, "trace_id": "tr-ea6a564e8cd9dc1c0f752ee0a4d9cf07"}


Trace(trace_id=tr-ea6a564e8cd9dc1c0f752ee0a4d9cf07)

## 10b. Conferir a proposta diretamente no PostgreSQL

Nova conex?o com APPLICATION_DATABASE_URL, ap?s o commit. Banco de neg?cio planejamento_db, tabelas planning.proposal_generations e planning.proposal_lines. **Esperado:** todas as linhas e valores iguais ? proposta, com run e vers?o corretas. Consulta versionada em sql/queries/verify_proposal.sql; esta etapa somente l?.


**Esperado/evidência:** PASS; evid?ncia exibida.


In [12]:
with step("verify proposal SQL", execution_id, statuses):
    database_evidence = verify_proposal_sql(cfg, execution_id, trace, lines)
    display(database_evidence)

{"timestamp": "2026-11-04T19:24:24.744319+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "verify proposal SQL", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


{'database': 'planejamento_db',
 'schema': 'planning',
 'status': 'PASS',
 'execution_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'run_id': 'c0d9f3138c874049b339e0f54ab8348c',
 'model_version': '15',
 'proposal_status': 'DRAFT',
 'stored_rows': 90,
 'proposed_students': 8694,
 'proposed_groups': 270,
 'query': 'sql/queries/verify_proposal.sql'}

{"timestamp": "2026-11-04T19:24:24.811749+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "verify proposal SQL", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 0.067, "attempt": 1, "trace_id": "tr-d90a4428170918659ad87a1b078ceb52"}


Trace(trace_id=tr-d90a4428170918659ad87a1b078ceb52)

## 11. Consultar API e reconciliar

A API consulta PostgreSQL sem treinamento. Comparar todas as linhas e rastreabilidade com a proposta escrita pelo notebook.

**Esperado/evidência:** Mesmos IDs, quantidades e total no SQL e na API; run ligada à geração.


In [13]:
with step("API reconciliation", execution_id, statuses):
    headers = {"Authorization":"Bearer " + required("API_TOKEN"), "X-Correlation-ID":execution_id, **trace_headers()}
    def get_api(path):
        r = httpx.get(cfg.api + path, headers={**headers, **trace_headers()}, timeout=cfg.timeout)
        r.raise_for_status()
        return r.json()
    api_generation = get_api(f"/v1/proposals/{execution_id}")
    api_lines = get_api(f"/v1/proposals/{execution_id}/lines?limit=500")
    assert api_generation["run_id"] == trace["run_id"]
    assert api_generation["model_version"] == trace["model_version"]
    assert len(api_lines) == len(lines) == api_generation["row_count"]
    expected = {tuple(r[k] for k in ["course_id","period_id","discipline_id"]):r for r in lines}
    for row in api_lines:
        original = expected[tuple(row[k] for k in ["course_id","period_id","discipline_id"])]
        assert row["proposed_students"] == original["proposed_students"]
        assert row["proposed_groups"] == original["proposed_groups"]
        assert row["forecast_raw"] == original["forecast_raw"]
    display(api_generation)
    display(pd.DataFrame(api_lines).head())

{"timestamp": "2026-11-04T19:24:24.837268+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "API reconciliation", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88"}


{'execution_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'correlation_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'course_scope': None,
 'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
 'payload_sha256': '406a0f2c6047a48db409ae4a58f3c3cbe2a84294adbfc03a8e5bb5261234bed9',
 'run_id': 'c0d9f3138c874049b339e0f54ab8348c',
 'experiment_id': '4',
 'model_name': 'planilha-turmas-demo',
 'model_version': '15',
 'artifact_uri': 'models:/m-342590e0c9ca494e99e6d746160225f8',
 'rule_version': 'POC-whole-cohort-ceil-capacity-v1',
 'capacity': 40,
 'mode': 'DEMO',
 'status': 'DRAFT',
 'row_count': 90,
 'generated_at': '2026-11-04T19:24:24.629371+00:00'}

,execution_id,course_id,period_id,discipline_id,forecast_raw,proposed_students,proposed_groups,human_adjustment
0,712b11cd-a96a-4bb9-83bb-691724ceae88,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,102.089955,103,3,None
1,712b11cd-a96a-4bb9-83bb-691724ceae88,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,102.089955,103,3,None
2,712b11cd-a96a-4bb9-83bb-691724ceae88,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,102.089955,103,3,None
3,712b11cd-a96a-4bb9-83bb-691724ceae88,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,103.245511,104,3,None
4,712b11cd-a96a-4bb9-83bb-691724ceae88,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,103.245511,104,3,None


{"timestamp": "2026-11-04T19:24:25.200228+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "API reconciliation", "correlation_id": "712b11cd-a96a-4bb9-83bb-691724ceae88", "duration_seconds": 0.363, "attempt": 1, "trace_id": "tr-1ba0a12ba58ff329b3e4a8b3b2dd6b96"}


Trace(trace_id=tr-1ba0a12ba58ff329b3e4a8b3b2dd6b96)

## 12. Evidências finais

Salvar resumo reproduzível da execução no workspace e no MLflow. Consultar dashboard pelo correlation_id.

**Esperado/evidência:** Todas as etapas PASS, evidência local e no Artifact Store, links sem segredos.


In [14]:
tracing = trace_evidence(cfg, statuses, trace, execution_id)
display(pd.DataFrame(tracing))
summary = {"execution_id":execution_id, "mode":cfg.mode, "sql_rows":len(history), "metrics":metrics,
    **trace, "proposal_id":execution_id, "persisted_rows":len(lines), "api_rows":len(api_lines),
    "snapshot_id":snapshot["id"], "dataset_sha256":snapshot["dataset_sha256"], "stages":statuses, "traces":tracing, "storage_evidence":storage_evidence, "database_evidence":database_evidence}
reports = Path("/workspace/reports")
reports.mkdir(parents=True, exist_ok=True)
report = reports / f"journey-{execution_id}.json"
report.write_text(json.dumps(summary, indent=2), encoding="utf-8")
mlflow.MlflowClient().log_artifact(trace["run_id"], str(report), "evidence")
display(pd.DataFrame(statuses))
display(summary)
display(Markdown(f"[Run MLflow]({trace['run_url']}) · [API](http://127.0.0.1:8000/docs) · [Dashboard](http://127.0.0.1:8088) · [RustFS](http://127.0.0.1:9001)"))

,stage,attempt,trace_id,spans,url
0,SQL extraction,1,tr-4222edf555191c6c97a244cb6330e788,[SQL extraction],http://127.0.0.1:5000/#/experiments/4/traces?s...
1,prepare,1,tr-e3bbd67a52793a1edf81fca8c6fa2614,[prepare],http://127.0.0.1:5000/#/experiments/4/traces?s...
2,model and metrics,1,tr-0ea4a6b4b33d6cb4ef6e01df7e3db1f6,[model and metrics],http://127.0.0.1:5000/#/experiments/4/traces?s...
3,tracking,1,tr-42410e8fafa17c97720af61ba16bc7c7,[tracking],http://127.0.0.1:5000/#/experiments/4/traces?s...
4,fit and evaluate,1,tr-6729e65a3bdc382eccf04fa0dd3d74ce,[fit and evaluate],http://127.0.0.1:5000/#/experiments/4/traces?s...
5,register,1,tr-1c61032c4f969e14529ed85397d18177,[register],http://127.0.0.1:5000/#/experiments/4/traces?s...
6,verify Object Store,1,tr-1885e64cc3ee4d33e08f9eb6e6c11d90,[verify Object Store],http://127.0.0.1:5000/#/experiments/4/traces?s...
7,reload and inference,1,tr-f94aea0befa7ae596ba1c43bd3fc8d94,"[reload and inference, model.load, inference.h...",http://127.0.0.1:5000/#/experiments/4/traces?s...
8,compose proposal,1,tr-aefa5756077a436d82fecfd549b42358,[compose proposal],http://127.0.0.1:5000/#/experiments/4/traces?s...
9,persist and idempotency,1,tr-ea6a564e8cd9dc1c0f752ee0a4d9cf07,[persist and idempotency],http://127.0.0.1:5000/#/experiments/4/traces?s...


,stage,status,seconds,attempt,trace_id
0,dependencies,PASS,NaN,NaN,NaN
1,SQL extraction,PASS,0.199,1.0,tr-4222edf555191c6c97a244cb6330e788
2,prepare,PASS,0.045,1.0,tr-e3bbd67a52793a1edf81fca8c6fa2614
3,model and metrics,PASS,0.044,1.0,tr-0ea4a6b4b33d6cb4ef6e01df7e3db1f6
4,tracking,PASS,0.152,1.0,tr-42410e8fafa17c97720af61ba16bc7c7
5,fit and evaluate,PASS,0.097,1.0,tr-6729e65a3bdc382eccf04fa0dd3d74ce
6,register,PASS,1.159,1.0,tr-1c61032c4f969e14529ed85397d18177
7,verify Object Store,PASS,0.291,1.0,tr-1885e64cc3ee4d33e08f9eb6e6c11d90
8,reload and inference,PASS,1.039,1.0,tr-f94aea0befa7ae596ba1c43bd3fc8d94
9,compose proposal,PASS,0.044,1.0,tr-aefa5756077a436d82fecfd549b42358


{'execution_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'mode': 'DEMO',
 'sql_rows': 1050,
 'metrics': {'mae': 3.758630658273197, 'rmse': 4.9142272796917865},
 'run_id': 'c0d9f3138c874049b339e0f54ab8348c',
 'experiment_id': '4',
 'run_url': 'http://127.0.0.1:5000/#/experiments/4/runs/c0d9f3138c874049b339e0f54ab8348c',
 'model_name': 'planilha-turmas-demo',
 'model_version': '15',
 'model_uri': 'models:/planilha-turmas-demo/15',
 'artifact_uri': 'models:/m-342590e0c9ca494e99e6d746160225f8',
 'inference_id': 'be5cf3b8-2839-4658-bc50-e8d24d4c2add',
 'artifact_sha256': '4268666504b258b661b9c6cc009bdf8c21d06fde4317fb389a585e633be3ad0b',
 'proposal_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'persisted_rows': 90,
 'api_rows': 90,
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
 'stages': [{'stage': 'dependencies', 'status': 'PASS'},
  {'stage': 'SQL extraction',
   'status': 'PASS',
   'seconds

[Run MLflow](http://127.0.0.1:5000/#/experiments/4/runs/c0d9f3138c874049b339e0f54ab8348c) · [API](http://127.0.0.1:8000/docs) · [Dashboard](http://127.0.0.1:8088) · [RustFS](http://127.0.0.1:9001)